Установка и загрузка UDPipe модели

In [3]:
import wget
import os
import sys
from ufal.udpipe import Model, Pipeline

udpipe_url = 'https://rusvectores.org/static/models/udpipe_syntagrus.model'
modelfile = 'udpipe_syntagrus.model'

if not os.path.isfile(modelfile):
    print("Модель не найдена, скачиваем...")
    wget.download(udpipe_url)
    print("\nМодель загружена")
else:
    print("Модель уже загружена")

Модель уже загружена


Предобработка корпуса

In [5]:
import re

def process(pipeline, text='Строка', keep_pos=True, keep_punct=False):
    entities = {'PROPN'}
    named = False
    memory = []
    mem_case = None
    mem_number = None
    tagged_propn = []
    
    # обрабатываем текст, получаем результат в формате conllu:
    processed = pipeline.process(text)
    
    # пропускаем строки со служебной информацией:
    content = [l for l in processed.split('\n') if not l.startswith('#')]
    
    # извлекаем из обработанного текста леммы, тэги и морфологические характеристики
    tagged = [w.split('\t') for w in content if w]

    for t in tagged:
        if len(t) != 10:
            continue
        (word_id, token, lemma, pos, xpos, feats, head, deprel, deps, misc) = t
        if not lemma or not token:
            continue
        if pos in entities:
            if '|' not in feats:
                tagged_propn.append(f'{lemma}_{pos}')
                continue
            morph = {el.split('=')[0]: el.split('=')[1] for el in feats.split('|')}
            if 'Case' not in morph or 'Number' not in morph:
                tagged_propn.append(f'{lemma}_{pos}')
                continue
            if not named:
                named = True
                mem_case = morph['Case']
                mem_number = morph['Number']
            if morph['Case'] == mem_case and morph['Number'] == mem_number:
                memory.append(lemma)
                if 'SpacesAfter=\\n' in misc:
                    named = False
                    past_lemma = '::'.join(memory)
                    memory = []
                    tagged_propn.append(past_lemma + '_PROPN')
            else:
                named = False
                past_lemma = '::'.join(memory)
                memory = []
                tagged_propn.append(past_lemma + '_PROPN')
                tagged_propn.append(f'{lemma}_{pos}')
        else:
            if not named:
                if pos == 'NUM' and token.isdigit(): # Заменяем числа на xxxxx той же длины
                    continue
                tagged_propn.append(f'{lemma}_{pos}')
            else:
                named = False
                past_lemma = '::'.join(memory)
                memory = []
                tagged_propn.append(past_lemma + '_PROPN')
                tagged_propn.append(f'{lemma}_{pos}')

    if not keep_punct:
        tagged_propn = [word for word in tagged_propn if word.split('_')[1] != 'PUNCT']
    if not keep_pos:
        tagged_propn = [word.split('_')[0] for word in tagged_propn]
    return tagged_propn

In [6]:
def tag_ud(text, modelfile):
    model = Model.load(modelfile)
    process_pipeline = Pipeline(model, 'tokenize', Pipeline.DEFAULT, Pipeline.DEFAULT, 'conllu')
    lines = text.split('\n')
    tagged = []
    for line in lines:
        output = process(process_pipeline, text=line)
        tagged_line = ' '.join(output)
        tagged.append(tagged_line)
    return '\n'.join(tagged)

Предобработка корпусов

In [7]:
files = ["prose_clean.txt"]  

for file in files:
    text = open(file, 'r', encoding='utf-8').read()
    processed = tag_ud(text, modelfile)
    with open(f"processed_{file}", 'w', encoding='utf-8') as out:
        out.write(processed)
    print(f"Обработан: {file}")

Обработан: prose_clean.txt


Обучение моделей CBOW и Skip-Gram

In [9]:
from gensim.models import Word2Vec
from gensim.models.word2vec import LineSentence

In [10]:
for file in files:
    corpus = f"processed_{file}"
    data = LineSentence(corpus)

    model_cb = Word2Vec(data, vector_size=500, window=5, min_count=2, sg=0)
    model_sg = Word2Vec(data, vector_size=500, window=5, min_count=2, sg=1)

    model_cb.save(f"cbow_{file}.model")
    model_sg.save(f"skipgram_{file}.model")
    print(f"Сохранены модели для: {file}")

Сохранены модели для: prose_clean.txt


Сравнение CBOW и Skip-Gram моделей

In [11]:
from gensim.models import Word2Vec
import numpy as np
from IPython.display import display
import pandas as pd

In [12]:
word_pairs = [("экзамен_NOUN", "государственный_ADJ"),
              ("командировка_NOUN", "капитан_NOUN"),
              ("командировка_NOUN", "уехать_VERB")]

In [13]:
for file in files:
    cbow = Word2Vec.load(f"cbow_{file}.model")
    sg = Word2Vec.load(f"skipgram_{file}.model")
    print(f"\nСравнение моделей для корпуса: {file}")
    rows = []
    for w1, w2 in word_pairs:
        try:
            cos_cb = cbow.wv.similarity(w1, w2)
            cos_sg = sg.wv.similarity(w1, w2)
            euc_cb = np.linalg.norm(cbow.wv[w1] - cbow.wv[w2])
            euc_sg = np.linalg.norm(sg.wv[w1] - sg.wv[w2])
            rows.append([f"{w1} vs {w2}", cos_cb, cos_sg, euc_cb, euc_sg])
        except KeyError:
            print(f"⚠️ Слова не найдены в словаре: {w1}, {w2}")
    df = pd.DataFrame(rows, columns=["Пара слов", "Косинус CBOW", "Косинус SG", "Евклид CBOW", "Евклид SG"])
    display(df)


Сравнение моделей для корпуса: prose_clean.txt


,Пара слов,Косинус CBOW,Косинус SG,Евклид CBOW,Евклид SG
0,экзамен_NOUN vs государственный_ADJ,0.942378,0.893551,0.801963,0.870775
1,командировка_NOUN vs капитан_NOUN,0.652721,0.571767,3.709548,2.243612
2,командировка_NOUN vs уехать_VERB,0.860336,0.933930,1.105500,0.829208
